# 05 — Fine-tune flan-T5-small for Text↔Gloss

**Two models, one notebook** (same architecture, opposite direction):
1. `t5_text2gloss/`  — English → ASL gloss (used by Text→Sign and Speech→Sign)
2. `gloss2text/`     — ASL gloss → English sentence (used to enrich Sign→Text output)

**Training data, in order of importance:**
1. ASLG-PC12 (~87k synthetic English↔gloss pairs) — bootstrap.
2. How2Sign (~35k continuous-signing sentence/gloss pairs) — refinement.

**Foolproof:** if this notebook fails entirely, the runtime falls back to the
rule-based heuristic in `ml-service/app/inference/{text_to_gloss,gloss_to_text}.py`.

**Compute:** Colab T4, ~60 min for both directions.

**Outputs (Drive + HF):**
- `models/t5_text2gloss/`  + ONNX-int8 export
- `models/gloss2text/`      + ONNX-int8 export

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/dl_project'
import os
os.environ.setdefault('HF_TOKEN', 'hf_xxx_paste_here')
HF_ORG = 'uet-signlang'

!pip install -q 'transformers>=4.45' 'datasets>=2.20' accelerate sentencepiece sacrebleu \
    'optimum[onnxruntime]>=1.22' onnxruntime huggingface_hub

In [ ]:
# Load ASLG-PC12 (87k synthetic pairs).
from datasets import load_dataset
aslg = load_dataset('achrafothman/aslg_pc12')
print(aslg)
print(aslg['train'][0])

# Normalise into uniform {'src': str, 'tgt': str} pairs.
TEXT_KEY = 'text' if 'text' in aslg['train'].column_names else aslg['train'].column_names[0]
GLOSS_KEY = 'gloss' if 'gloss' in aslg['train'].column_names else aslg['train'].column_names[1]
print('using src=', TEXT_KEY, ' tgt=', GLOSS_KEY)

def to_pairs(rec):
    return {'src': rec[TEXT_KEY].strip().lower(), 'tgt': rec[GLOSS_KEY].strip().upper()}

aslg_pairs = aslg['train'].map(to_pairs, remove_columns=aslg['train'].column_names)
aslg_pairs = aslg_pairs.train_test_split(test_size=0.02, seed=42)
print(aslg_pairs)

In [ ]:
# (Optional) Append How2Sign text/gloss CSV pairs if you've placed them on Drive
# at  data/how2sign/{train,val,test}.csv  with columns SENTENCE,GLOSS.
import os, pandas as pd
from datasets import Dataset

H2S_DIR = f'{BASE}/data/how2sign'
h2s_train = h2s_dev = None
if os.path.exists(f'{H2S_DIR}/train.csv'):
    df_tr = pd.read_csv(f'{H2S_DIR}/train.csv')
    df_tr = df_tr.dropna(subset=['SENTENCE', 'GLOSS'])
    h2s_train = Dataset.from_pandas(pd.DataFrame({
        'src': df_tr['SENTENCE'].str.strip().str.lower(),
        'tgt': df_tr['GLOSS'].str.strip().str.upper(),
    }))
    print('How2Sign train:', len(h2s_train))
if os.path.exists(f'{H2S_DIR}/val.csv'):
    df_dv = pd.read_csv(f'{H2S_DIR}/val.csv').dropna(subset=['SENTENCE', 'GLOSS'])
    h2s_dev = Dataset.from_pandas(pd.DataFrame({
        'src': df_dv['SENTENCE'].str.strip().str.lower(),
        'tgt': df_dv['GLOSS'].str.strip().str.upper(),
    }))
    print('How2Sign dev:', len(h2s_dev))
else:
    print('No How2Sign CSVs found at', H2S_DIR, '- training on ASLG-PC12 only.')

In [ ]:
from transformers import AutoTokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments, DataCollatorForSeq2Seq
import torch, os, json

BASE_MODEL = 'google/flan-t5-small'
MAX_SRC, MAX_TGT = 96, 48

def train_one(direction: str, prompt: str, out_dir: str):
    """direction is 'text2gloss' or 'gloss2text'."""
    tok = AutoTokenizer.from_pretrained(BASE_MODEL)
    model = T5ForConditionalGeneration.from_pretrained(BASE_MODEL)

    def encode(ex):
        src = ex['src'] if direction == 'text2gloss' else ex['tgt']
        tgt = ex['tgt'] if direction == 'text2gloss' else ex['src']
        m = tok(prompt + src, max_length=MAX_SRC, truncation=True)
        with tok.as_target_tokenizer():
            l = tok(tgt, max_length=MAX_TGT, truncation=True)
        m['labels'] = l['input_ids']
        return m

    tr_a = aslg_pairs['train'].map(encode, remove_columns=aslg_pairs['train'].column_names)
    ev_a = aslg_pairs['test'].map( encode, remove_columns=aslg_pairs['test'].column_names)

    args_a = TrainingArguments(output_dir=f'{out_dir}/_stageA', num_train_epochs=3,
        per_device_train_batch_size=16, per_device_eval_batch_size=16,
        learning_rate=3e-4, lr_scheduler_type='cosine', warmup_ratio=0.05, fp16=True,
        eval_strategy='epoch', save_strategy='epoch', save_total_limit=2,
        load_best_model_at_end=True, metric_for_best_model='loss',
        report_to='none', seed=42)
    Trainer(model=model, args=args_a, train_dataset=tr_a, eval_dataset=ev_a,
            data_collator=DataCollatorForSeq2Seq(tok, model=model)).train(resume_from_checkpoint=False)

    if h2s_train is not None:
        tr_b = h2s_train.map(encode, remove_columns=h2s_train.column_names)
        ev_b = h2s_dev.map(encode,  remove_columns=h2s_dev.column_names) if h2s_dev is not None else ev_a
        args_b = TrainingArguments(output_dir=f'{out_dir}/_stageB', num_train_epochs=2,
            per_device_train_batch_size=16, per_device_eval_batch_size=16,
            learning_rate=1e-4, lr_scheduler_type='cosine', warmup_ratio=0.05, fp16=True,
            eval_strategy='epoch', save_strategy='epoch', save_total_limit=2,
            load_best_model_at_end=True, metric_for_best_model='loss',
            report_to='none', seed=42)
        Trainer(model=model, args=args_b, train_dataset=tr_b, eval_dataset=ev_b,
                data_collator=DataCollatorForSeq2Seq(tok, model=model)).train()

    os.makedirs(out_dir, exist_ok=True)
    model.save_pretrained(out_dir); tok.save_pretrained(out_dir)
    return tok, model

T2G_DIR = f'{BASE}/models/t5_text2gloss'
G2T_DIR = f'{BASE}/models/gloss2text'
tok_t2g, m_t2g = train_one('text2gloss', 'translate English to ASL gloss: ', T2G_DIR)
tok_g2t, m_g2t = train_one('gloss2text', 'translate ASL gloss to English: ', G2T_DIR)

In [ ]:
# BLEU-4 on How2Sign dev (or ASLG dev as fallback).
import sacrebleu

def bleu(model, tok, prompt, eval_ds, src_key, tgt_key, n_max=500):
    preds, refs = [], []
    model.eval()
    for ex in eval_ds.select(range(min(n_max, len(eval_ds)))):
        ids = tok(prompt + ex[src_key], return_tensors='pt').input_ids.to(model.device)
        out = model.generate(ids, max_length=MAX_TGT, num_beams=4)
        preds.append(tok.decode(out[0], skip_special_tokens=True))
        refs.append(ex[tgt_key])
    return sacrebleu.corpus_bleu(preds, [refs]).score

if h2s_dev is not None:
    score_t2g = bleu(m_t2g, tok_t2g, 'translate English to ASL gloss: ', h2s_dev, 'src', 'tgt')
    score_g2t = bleu(m_g2t, tok_g2t, 'translate ASL gloss to English: ', h2s_dev, 'tgt', 'src')
else:
    score_t2g = bleu(m_t2g, tok_t2g, 'translate English to ASL gloss: ', aslg_pairs['test'], 'src', 'tgt')
    score_g2t = bleu(m_g2t, tok_g2t, 'translate ASL gloss to English: ', aslg_pairs['test'], 'tgt', 'src')
print(f'BLEU-4 text->gloss: {score_t2g:.2f}')
print(f'BLEU-4 gloss->text: {score_g2t:.2f}')

In [ ]:
# Quantise + ONNX export for cheap CPU serving.
from optimum.onnxruntime import ORTModelForSeq2SeqLM, ORTQuantizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig
import os

def export_int8(src_dir):
    onnx_dir = f'{src_dir}/onnx_int8'
    os.makedirs(onnx_dir, exist_ok=True)
    ort_model = ORTModelForSeq2SeqLM.from_pretrained(src_dir, export=True)
    ort_model.save_pretrained(onnx_dir)
    qcfg = AutoQuantizationConfig.avx2(is_static=False, per_channel=True)
    for sub in ['encoder_model.onnx', 'decoder_model.onnx', 'decoder_with_past_model.onnx']:
        p = f'{onnx_dir}/{sub}'
        if os.path.exists(p):
            ORTQuantizer.from_pretrained(onnx_dir, file_name=sub).quantize(
                save_dir=onnx_dir, quantization_config=qcfg)
    print('int8 export ready in', onnx_dir)

export_int8(T2G_DIR)
export_int8(G2T_DIR)

In [ ]:
from huggingface_hub import HfApi, create_repo
for repo, src in [(f'{HF_ORG}/t5-text2gloss', T2G_DIR),
                  (f'{HF_ORG}/t5-gloss2text', G2T_DIR)]:
    create_repo(repo, repo_type='model', private=True, exist_ok=True)
    HfApi().upload_folder(folder_path=src, repo_id=repo, repo_type='model')
    print('Pushed', repo)